# 04 - Pipeline: Cálculo do IVU por Bairro

Une as três dimensões (renda, segurança, mobilidade), calcula o Índice de Vulnerabilidade Urbana (IVU) por bairro e exporta os arquivos finais para o dashboard.

**Fórmula:**
$$\text{IVU} = (\text{nota\_renda} \times 0{,}4) + (\text{nota\_seguranca} \times 0{,}3) + (\text{nota\_mobilidade} \times 0{,}3)$$

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import plotly.express as px


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()


ROOT = find_root()

NOTAS_RENDA     = ROOT / 'data' / 'processed' / 'notas_renda.csv'
NOTAS_SEGURANCA = ROOT / 'data' / 'processed' / 'notas_seguranca.csv'
NOTAS_MOBILIDADE = ROOT / 'data' / 'processed' / 'mobilidade_por_bairro.csv'
GEOJSON_BAIRROS  = ROOT / 'data' / 'processed' / 'recife_mobilidade.geojson'

OUTPUT_CSV     = ROOT / 'data' / 'final' / 'ivu_final.csv'
OUTPUT_GEOJSON = ROOT / 'data' / 'final' / 'recife_ivu.geojson'

PESOS = {'renda': 0.4, 'seguranca': 0.3, 'mobilidade': 0.3}

print(f'Raiz: {ROOT}')
print(f'Pesos: renda={PESOS["renda"]} | seguranca={PESOS["seguranca"]} | mobilidade={PESOS["mobilidade"]}')

## 1 - Carregar os 3 arquivos de notas

In [ ]:
renda     = pd.read_csv(NOTAS_RENDA)
seguranca = pd.read_csv(NOTAS_SEGURANCA)
mobilidade = pd.read_csv(NOTAS_MOBILIDADE)

for nome, df in [('renda', renda), ('seguranca', seguranca), ('mobilidade', mobilidade)]:
    assert list(df.columns) == ['bairro', 'nota_dimensao', 'dado_principal'], \
        f'Colunas inesperadas em {nome}: {df.columns.tolist()}'
    assert len(df) == 94, f'{nome} tem {len(df)} linhas (esperado 94)'
    assert df.isnull().sum().sum() == 0, f'{nome} tem valores nulos'
    assert df['bairro'].nunique() == 94, f'{nome} tem bairros duplicados'

print('Arquivos carregados e validados:')
print(f'  renda     : {len(renda)} bairros')
print(f'  seguranca : {len(seguranca)} bairros')
print(f'  mobilidade: {len(mobilidade)} bairros')

## 2 - Merge e cálculo do IVU

In [ ]:
df = renda.rename(columns={'nota_dimensao': 'nota_renda', 'dado_principal': 'dado_renda'})
df = df.merge(
    seguranca.rename(columns={'nota_dimensao': 'nota_seguranca', 'dado_principal': 'dado_seguranca'}),
    on='bairro',
)
df = df.merge(
    mobilidade.rename(columns={'nota_dimensao': 'nota_mobilidade', 'dado_principal': 'dado_mobilidade'}),
    on='bairro',
)

df['IVU'] = (
    df['nota_renda']     * PESOS['renda']     +
    df['nota_seguranca'] * PESOS['seguranca'] +
    df['nota_mobilidade'] * PESOS['mobilidade']
).round(2)

df = df.sort_values('IVU', ascending=False).reset_index(drop=True)
df.index += 1

print(f'Merge concluido: {len(df)} bairros')
print(f'IVU  min: {df["IVU"].min():.2f} | max: {df["IVU"].max():.2f} | media: {df["IVU"].mean():.2f}')
display(
    df[['bairro', 'nota_renda', 'nota_seguranca', 'nota_mobilidade', 'IVU']]
    .head(10)
    .style
    .format({'nota_renda': '{:.2f}', 'nota_seguranca': '{:.2f}', 'nota_mobilidade': '{:.2f}', 'IVU': '{:.2f}'})
    .background_gradient(subset=['IVU'], cmap='RdYlGn')
    .hide(axis='index')
)

## 3 - Validação do resultado

In [ ]:
assert len(df) == 94, f'Merge perdeu bairros: {len(df)} (esperado 94)'
assert df['IVU'].between(0, 10).all(), 'IVU fora do intervalo [0, 10]'
assert df.isnull().sum().sum() == 0, 'Valores nulos no dataframe final'
assert df['bairro'].nunique() == 94, 'Bairros duplicados no resultado'

soma_pesos = sum(PESOS.values())
assert abs(soma_pesos - 1.0) < 1e-9, f'Pesos nao somam 1: {soma_pesos}'

print('Todas as validacoes passaram:')
print(f'  94 bairros sem nulos')
print(f'  IVU no intervalo [{df["IVU"].min():.2f}, {df["IVU"].max():.2f}]')
print(f'  Pesos somam {soma_pesos:.1f}')

## 4 - Ranking dos bairros

In [ ]:
top10    = df.nlargest(10, 'IVU')[['bairro', 'IVU', 'nota_renda', 'nota_seguranca', 'nota_mobilidade']].reset_index(drop=True)
bottom10 = df.nsmallest(10, 'IVU')[['bairro', 'IVU', 'nota_renda', 'nota_seguranca', 'nota_mobilidade']].reset_index(drop=True)
top10.index    += 1
bottom10.index += 1

fmt = {'IVU': '{:.2f}', 'nota_renda': '{:.2f}', 'nota_seguranca': '{:.2f}', 'nota_mobilidade': '{:.2f}'}

print('Top 10 — maior IVU (menos vulneraveis):')
display(top10.style.format(fmt).background_gradient(subset=['IVU'], cmap='Greens').hide(axis='index'))

print('Bottom 10 — menor IVU (mais vulneraveis):')
display(bottom10.style.format(fmt).background_gradient(subset=['IVU'], cmap='Reds_r').hide(axis='index'))

In [ ]:
fig = px.bar(
    df.sort_values('IVU'),
    x='IVU',
    y='bairro',
    orientation='h',
    color='IVU',
    color_continuous_scale='RdYlGn',
    range_color=[0, 10],
    text='IVU',
    hover_data={'nota_renda': ':.2f', 'nota_seguranca': ':.2f', 'nota_mobilidade': ':.2f'},
    labels={'IVU': 'IVU (0-10)', 'bairro': 'Bairro'},
    title='Índice de Vulnerabilidade Urbana por Bairro — Recife (2025)',
    height=2000,
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    margin=dict(l=10, r=60, t=50, b=10),
    coloraxis_showscale=False,
    xaxis=dict(range=[0, 11]),
)
fig.show()

## 5 - Join com GeoJSON dos bairros

In [ ]:
gdf = gpd.read_file(GEOJSON_BAIRROS)

gdf_ivu = gdf[['NM_BAIRRO', 'geometry']].merge(
    df.rename(columns={'bairro': 'NM_BAIRRO'}),
    on='NM_BAIRRO',
    how='left',
)

sem_ivu = gdf_ivu[gdf_ivu['IVU'].isna()]['NM_BAIRRO'].tolist()
if sem_ivu:
    print(f'ATENCAO: {len(sem_ivu)} bairros sem IVU no GeoJSON: {sem_ivu}')
else:
    print(f'Join perfeito: {len(gdf_ivu)} bairros com IVU calculado')

display(
    gdf_ivu[['NM_BAIRRO', 'IVU', 'nota_renda', 'nota_seguranca', 'nota_mobilidade']]
    .sort_values('IVU', ascending=False)
    .head(10)
    .reset_index(drop=True)
    .style
    .format({'IVU': '{:.2f}', 'nota_renda': '{:.2f}', 'nota_seguranca': '{:.2f}', 'nota_mobilidade': '{:.2f}'})
    .background_gradient(subset=['IVU'], cmap='RdYlGn')
    .hide(axis='index')
)

## 6 - Salvar arquivos finais

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

cols_csv = [
    'bairro', 'IVU',
    'nota_renda', 'nota_seguranca', 'nota_mobilidade',
    'dado_renda', 'dado_seguranca', 'dado_mobilidade',
]
df_out = df.rename(columns={'NM_BAIRRO': 'bairro'}) if 'NM_BAIRRO' in df.columns else df
df_out[cols_csv].to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

gdf_ivu.to_file(OUTPUT_GEOJSON, driver='GeoJSON')

print(f'ivu_final.csv  salvo: {OUTPUT_CSV}')
print(f'recife_ivu.geojson salvo: {OUTPUT_GEOJSON}')
print(f'Total de bairros: {len(df_out)}')
display(
    df_out[cols_csv].head(10)
    .reset_index(drop=True)
    .style
    .format({'IVU': '{:.2f}', 'nota_renda': '{:.2f}', 'nota_seguranca': '{:.2f}', 'nota_mobilidade': '{:.2f}'})
    .background_gradient(subset=['IVU'], cmap='RdYlGn')
    .hide(axis='index')
)